<div style="background: linear-gradient(120deg,#0b1f3a 0%,#173f73 55%,#2a7ab8 100%); color: #ffffff; padding: 26px 30px; border-radius: 14px; font-family: 'Segoe UI', Roboto, Arial, sans-serif;">
<div style="font-size: 30px; font-weight: 700; line-height: 1.2; color: #ffffff;">QCC 27.307: the public frontier + a finer chemistry weight refit (v4)</div>
<div style="font-size: 15px; margin-top: 10px; color: #ffffff;">Google Verifiable Quantum Circuit Compilation &middot; every file re-verified with the unchanged official verifiers &middot; full credits below</div>
<div style="margin-top: 14px;"><span style="background: #ffd166; color: #0b1f3a; padding: 4px 10px; border-radius: 999px; font-weight: 700;">GM 27.3068 (local, default salt)</span>
<span style="background: #ffffff; color: #0b1f3a; padding: 4px 10px; border-radius: 999px; margin-left: 6px;">lambda_eff 46.905894 &rarr; 46.905863</span></div>
</div>

This notebook takes the best public file of each task (2026-11-14, maoxin123's v27 portfolio, which already carries
our invsqrt dead-write fix) and adds **one change of ours to the chemistry**: the DF-THC weights `W` and `H2_I` are
refitted at the same angle words and loaders, starting from Atakan Aldemir's latest weights (`e52b7cdd`) and lowering
`lambda_eff` a little further while every accuracy gate of the verifier still passes. Nothing else changes. If this notebook helps you, an upvote is appreciated.

| Task | File | Qubits | Built by (latest change first) |
|---|---|---:|---|
| invsqrt | `e46d90c3` | 46 | **our dead-write fix** on `dcde7078`, the public 46-qubit line (Syed Asad Ali's own-circuit invsqrt, Ali Haydar Özdağ's `4560ac0c`) |
| DQI | `821bf8d1` | 889 | Areeb Ali's 241 retained products on Syed Asad Ali's v12 decoder at extractor cutoff 18 (Ertuğrul Özer) |
| chemistry | maoxin123's table/words + 15 loaders `71735b23`, **weights refitted here** | 427 | maoxin123 (v27 loaders and coded words); weight line by Heliosli, Atakan Aldemir and prvsiyan's refit code; Lera (marculera)'s table line |

## Acknowledgements
Thank you to **maoxin123** (chemistry table, coded words and the new loaders), **Syed Asad Ali** (from-source builders),
**Ali Haydar Özdağ**, **Areeb Ali**, **Ertuğrul Özer**, **Atakan Aldemir**, **Heliosli**, **prvsiyan** (whose refit code
we ported to numpy and extended with a full-Newton restoration), **Lera (marculera)** and the authors her NOTICE credits,
**Alex Chilton**, **Kirill Tymchishin**, **SomeJoint**, **busyaprime**, **yoyobar**, **mrmeows**, the team **i want to run
gemma 9b**, **jday96314** and **Oleg Sher**; and Google Quantum AI and Kaggle for the competition, kickmix and the verifiers.

**What is ours here:** the chemistry weight refit (and, already in the public invsqrt file, our dead-write fix).
Companion notebooks: [QCC 27.30 | What Works, What Fails and Why](https://www.kaggle.com/code/asteriospanoras/qcc-28-06018-what-works-what-fails-and-why)
and [QCC 27.30 | Public Frontier + Free CZ Fix in invsqrt](https://www.kaggle.com/code/asteriospanoras/qcc-27-30-public-frontier-free-cz-fix-in-invsq).

<div style="background: #fff7e0; border-left: 6px solid #ffb703; padding: 12px 16px; border-radius: 8px; color: #1b1b1b;">
<b>Getting the submission file.</b> This notebook rebuilds and verifies everything when it runs. To get
<code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px;">submission.zip</code>: <b>Copy &amp; Edit</b>, set
<code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px;">WRITE_SUBMISSION = True</code> in the next cell and <b>Run All</b>
(about 15 minutes on CPU). Please keep the credits.
</div>

In [ ]:
WRITE_SUBMISSION = True
import sys, os, pathlib, zipfile, json, time, subprocess, hashlib, math, csv, io, lzma, base64
import numpy as np
T0 = time.time()
def log(*a): print(f"[{(time.time()-T0)/60:5.1f} min]", *a, flush=True)
inp = pathlib.Path("/kaggle/input")
KM = next(p for p in inp.rglob("kickmix") if p.is_dir() and (p.parent / "verifiers").is_dir())
VER = KM.parent / "verifiers"; sys.path.insert(0, str(KM))
sha = lambda b: hashlib.sha256(b).hexdigest()
SUB = pathlib.Path("/tmp/sub"); (SUB / "dfthcsos").mkdir(parents=True, exist_ok=True)
def _npz_sha(z):
    try:
        with zipfile.ZipFile(z) as Z: return sha(Z.read("dfthcsos/submission.npz"))
    except Exception: return ""
src = next(z for z in inp.rglob("submission.zip") if _npz_sha(z).startswith("3f864a7a"))
PIN = {"invsqrt.kmb": "e46d90c3", "dqi.kmb": "821bf8d1", "dfthcsos/submission.npz": "3f864a7a", "dfthcsos/angle_k_0.kmb": "71735b23"}
with zipfile.ZipFile(src) as Z:
    for n in Z.namelist():
        if n.endswith("/"): continue
        b = Z.read(n)
        if n in PIN: assert sha(b).startswith(PIN[n]), (n, sha(b)[:12])
        (SUB / n).parent.mkdir(parents=True, exist_ok=True); (SUB / n).write_bytes(b)
log("public files", {n: sha((SUB / n).read_bytes())[:8] for n in PIN})

## Our change: refitted chemistry weights at fixed words
The verifier's cost is `lambda_eff x T x Q`; at fixed angle words the loaders (and so `T` and `Q`) do not change, but
`lambda_eff` depends on the DF-THC weights. Starting from Atakan Aldemir's latest weights on maoxin123's table we minimised the one-norm under the
verifier's accuracy gates (error_1, error_2 <= 20 mHa, E_gap above its floor) with a generalised reduced-gradient refit
(prvsiyan's method, our numpy port with a full Newton restoration step), keeping a margin of at least 1.5e-9 Ha on each
gate. The refitted `W` and `H2_I` are embedded below (lzma + base64) and checked by hash; every other array is unchanged.

In [ ]:
REFIT_B64 = "REMOVED_EMBEDDED_BINARY"
REFIT_HASH = "309ccfe6150375919ea8b0d623a216243ae8df9975311d8fda6287b5f670be50"
new = np.load(io.BytesIO(lzma.decompress(base64.b64decode(REFIT_B64))))
assert sha(new["W"].tobytes() + new["H2_I"].tobytes()) == REFIT_HASH
old = np.load(SUB / "dfthcsos/submission.npz")
arrays = {k: np.array(old[k]) for k in old.files}
assert arrays["W"].shape == new["W"].shape and arrays["H2_I"].shape == new["H2_I"].shape
print("max |dW| =", float(np.abs(arrays["W"] - new["W"]).max()), " max |dH2_I| =", float(np.abs(arrays["H2_I"] - new["H2_I"]).max()))
arrays["W"] = np.array(new["W"]); arrays["H2_I"] = np.array(new["H2_I"])
with open(SUB / "dfthcsos/submission.npz", "wb") as f: np.savez(f, **arrays)
log("chemistry npz with refitted weights", sha((SUB / "dfthcsos/submission.npz").read_bytes())[:12])

## Official verification of the exact bytes that go into `submission.zip`

In [ ]:
def verify(args, timeout=4 * 3600):
    env = dict(os.environ, PYTHONPATH=f"{KM}:{VER}")
    r = subprocess.run([sys.executable, *map(str, args)], capture_output=True, text=True, env=env, cwd=str(VER), timeout=timeout)
    try: return json.loads(r.stdout[r.stdout.index("{"):])
    except Exception: return {"status": "ERROR", "stderr": r.stderr[-2000:]}
res = {}
res["dqi"] = verify([VER / "dqi/verify_submission_dqi.py", SUB / "dqi.kmb"])
log("dqi", {k: res["dqi"].get(k) for k in ("status", "score", "toffoli_count", "num_qubits", "value_errors", "phase_errors")})
res["dfthcsos"] = verify([VER / "dfthcsos/verify.py", SUB / "dfthcsos"])
log("chemistry", {k: res["dfthcsos"].get(k) for k in ("status", "score", "toffoli_count", "num_qubits", "lambda_eff")})
res["invsqrt"] = verify([VER / "invsqrt/verify_submission_invsqrt.py", SUB / "invsqrt.kmb"])
log("invsqrt", {k: res["invsqrt"].get(k) for k in ("status", "score", "num_qubits", "toffoli_count", "worst_error_over_tolerance")})
assert all(res[k].get("status") == "PASS" for k in res), res

In [ ]:
REF = {"invsqrt": 2076774.23, "dqi": 24439463583672.93, "dfthcsos": 3355599588.32}
r = {k: REF[k] / res[k]["score"] for k in REF}
print({k: round(x, 6) for k, x in r.items()}, "GM", round(math.prod(r.values()) ** (1 / 3), 5))
OUT_ZIP = "/kaggle/working/submission.zip" if WRITE_SUBMISSION else "/tmp/submission.zip"
with zipfile.ZipFile(OUT_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(SUB.rglob("*")):
        if p.is_file():
            zi = zipfile.ZipInfo(str(p.relative_to(SUB)), date_time=(2026, 11, 14, 0, 0, 0)); zi.compress_type = zipfile.ZIP_DEFLATED
            z.writestr(zi, p.read_bytes())
json.dump({"sha256": {str(p.relative_to(SUB)): sha(p.read_bytes()) for p in SUB.rglob("*") if p.is_file()}, "verify": res},
          open("/kaggle/working/receipt.json", "w"), indent=1, default=str)
log("submission.zip written to", OUT_ZIP)

## Where each file can be found (read from the QCC Circuit Atlas)
The [QCC Circuit Atlas](https://www.kaggle.com/datasets/asteriospanoras/qcc-circuit-atlas) lists every public circuit
file of this competition once, with the public notebooks and datasets that hold the same bytes. The source shown is
one of them (the first in scan order, not necessarily the author: see the credits below for who built each file).
Files newer than the atlas scan (2026-11-12) are marked as such.

In [ ]:
atlas = next((p for p in inp.rglob("circuits.csv") if "atlas" in str(p).lower()), None)
rows = {r["sha12"]: r for r in csv.DictReader(open(atlas))} if atlas else {}
print(f"{'file':28s} {'sha12':12s} {'found in (one of)':58s} {'owner':22s} {'#sources':>8s}")
for p in sorted(SUB.rglob("*")):
    if not p.is_file(): continue
    h = sha(p.read_bytes())[:12]; r = rows.get(h)
    if r: print(f"{str(p.relative_to(SUB)):28s} {h:12s} {r['first_source_ref'][:58]:58s} {r['author'][:22]:22s} {r['n_sources']:>8s}")
    else: print(f"{str(p.relative_to(SUB)):28s} {h:12s} {'(newer than the atlas scan)':58s}")

## Credits (who did what), with links
- **Chemistry**: [maoxin123](https://www.kaggle.com/code/maoxin123/qcc-v27-4038-toffoli-coded-word-chemistry) ((8,143,44) table,
  coded words, v27 loaders `71735b23`); the weight line by [Heliosli](https://www.kaggle.com/code/heliosli/the-dataset-is-public-now-enjoy)
  and [Atakan Aldemir](https://www.kaggle.com/code/atakanaldemir/qcc-gap-3-residue-repair-and-c43-native-compiler) with
  [prvsiyan's refit code](https://www.kaggle.com/datasets/prvsiyan/vqcc-verified-best-of-circuits-1106a) (Apache-2.0); Lera
  (marculera)'s DF-THC line and the authors her NOTICE credits (Ahmed Berat Özer, kozykappa, anazemcev, mrmeows,
  michaelrustad, phucthaiv02, somejoint, gengsr, qwertykeypadapple, lightningv08, syed3000, busyaprime). **Ours:** the
  weight refit in this notebook (version 4 starts from Atakan Aldemir's weights `e52b7cdd`).
- **invsqrt** `e46d90c3`: our dead-write fix ([QCC 27.30 | Public Frontier + Free CZ Fix](https://www.kaggle.com/code/asteriospanoras/qcc-27-30-public-frontier-free-cz-fix-in-invsq))
  on `dcde7078`: [Syed Asad Ali](https://www.kaggle.com/code/syed3000/vqcc-own-circuit-portfolio-notebook)'s 46-qubit line and
  [Ali Haydar Özdağ](https://www.kaggle.com/code/ahoirg/qcc-submit-verify-and-package)'s `4560ac0c`.
- **DQI** `821bf8d1`: [Areeb Ali](https://www.kaggle.com/code/areebbb/qai-fewer-toffolis-same-qubits-lb-27-11042) on Syed Asad
  Ali's v12 decoder at extractor cutoff 18 ([Ertuğrul Özer](https://www.kaggle.com/code/ozertuu/quantum-ai-27-083-vqcc-dqi-cutoff-17));
  the decoder lineage is credited in those notebooks.
- Tasks, kickmix and verifiers: Google Quantum AI / Qualtran. Each file keeps its authors' terms; this notebook's code is Apache-2.0.